# 03 - Real Data Verification

The model was trained on synthetic charts. Now we test it on real S&P 500 price data.

We slide a 60-candle window across the historical price series, render each window
as a chart image, and run the CNN on it. The key question is: does confidence drop
significantly on real data compared to synthetic? That gap is the domain shift we
expected and it is the main research finding of the project.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
import io
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torchvision import transforms, models
import yfinance as yf

# load checkpoints from Drive
try:
    from google.colab import drive
    drive.mount('/drive')
    CKPT_DIR    = Path('/drive/MyDrive/ECE176/checkpoints')
    RESULTS_DIR = Path('/content/results')
except ImportError:
    CKPT_DIR    = Path('.').resolve() / 'checkpoints'
    RESULTS_DIR = Path('.').resolve() / 'results'

os.makedirs(RESULTS_DIR / 'figures', exist_ok=True)

IMG_SIZE   = 224
WINDOW     = 60    # candles per window
STEP       = 5     # slide step
THRESHOLD  = 0.55  # confidence threshold for detection

CLASS_NAMES = [
    'head_and_shoulders', 'double_top', 'descending_triangle',
    'inv_head_and_shoulders', 'double_bottom', 'ascending_triangle', 'no_pattern',
]
N_CLASSES = len(CLASS_NAMES)

CLASS_DIRECTION = {
    'head_and_shoulders':     'bearish',
    'double_top':             'bearish',
    'descending_triangle':    'bearish',
    'inv_head_and_shoulders': 'bullish',
    'double_bottom':          'bullish',
    'ascending_triangle':     'bullish',
    'no_pattern':             'neutral',
}

device = torch.device('cuda' if torch.cuda.is_available() else
                       'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'Device      : {device}')
print(f'Checkpoints : {CKPT_DIR}')


In [ ]:
class PatternCNN(nn.Module):
    def __init__(self, n_classes=N_CLASSES, dropout=0.5):
        super().__init__()
        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )
        self.features = nn.Sequential(
            conv_block(3, 32), conv_block(32, 64), conv_block(64, 128),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128, 256),
            nn.ReLU(inplace=True), nn.Dropout(dropout), nn.Linear(256, n_classes),
        )
    def forward(self, x):
        return self.classifier(self.gap(self.features(x)))


model = PatternCNN().to(device)
model.load_state_dict(torch.load(CKPT_DIR / 'cnn_best.pt', map_location=device))
model.eval()
print('Model loaded!')


## Download Real S&P 500 Data

We use `yfinance` to pull 2 years of daily OHLC data for the S&P 500 (^GSPC).
Daily candles give us enough history to find multiple pattern occurrences.

In [ ]:
ticker = yf.Ticker('^GSPC')
df = ticker.history(period='2y')
df = df[['Open', 'High', 'Low', 'Close']].dropna()
df.index = df.index.tz_localize(None)

print(f'Downloaded {len(df)} daily candles')
print(f'From: {df.index[0].date()}  To: {df.index[-1].date()}')
print(df.tail())


In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df.index, df['Close'], linewidth=0.8, color='steelblue')
ax.set_title('S&P 500 — 2 Years Daily Close', fontsize=12)
ax.set_ylabel('Price')
ax.set_xlabel('Date')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures' / 'sp500_price.png', dpi=120, bbox_inches='tight')
plt.show()


## Sliding Window Inference

We slide a 60-candle window over the price series with a step of 5.
Each window is rendered as a candlestick chart and passed through the CNN.
We record the predicted class and confidence for every window.

In [ ]:
import matplotlib
matplotlib.use('Agg')

tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


def render_window(ohlc):
    fig, ax = plt.subplots(figsize=(2.24, 2.24), dpi=100)
    ax.set_facecolor('#0d0d0d')
    fig.patch.set_facecolor('#0d0d0d')
    for i, (o, h, l, c) in enumerate(ohlc):
        color = '#26a69a' if c >= o else '#ef5350'
        ax.plot([i, i], [l, h], color=color, linewidth=0.8)
        ax.add_patch(plt.Rectangle((i - 0.3, min(o, c)), 0.6,
                                    max(abs(c - o), 1e-9), color=color))
    ax.set_xlim(-1, len(ohlc))
    ax.axis('off')
    plt.tight_layout(pad=0)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)


ohlc_arr = df[['Open', 'High', 'Low', 'Close']].values
dates    = df.index.to_list()

results = []
with torch.no_grad():
    for start in range(0, len(ohlc_arr) - WINDOW, STEP):
        window = ohlc_arr[start:start + WINDOW]
        img    = tfm(render_window(window)).unsqueeze(0).to(device)
        probs  = torch.softmax(model(img), dim=1)[0].cpu().numpy()
        pred   = probs.argmax()
        conf   = probs[pred]
        results.append({
            'date':      dates[start + WINDOW - 1],
            'class':     CLASS_NAMES[pred],
            'direction': CLASS_DIRECTION[CLASS_NAMES[pred]],
            'conf':      conf,
            'probs':     probs,
        })

print(f'Processed {len(results)} windows')


## Confidence Distribution

On synthetic data the model was basically always 100% confident.
On real data we expect much lower confidence — that gap is the domain shift.

In [ ]:
confs = [r['conf'] for r in results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# histogram
ax1.hist(confs, bins=30, color='steelblue', edgecolor='white', linewidth=0.5)
ax1.axvline(THRESHOLD, color='red', linestyle='--', label=f'threshold={THRESHOLD}')
ax1.axvline(np.mean(confs), color='orange', linestyle='--', label=f'mean={np.mean(confs):.2f}')
ax1.set_title('Confidence Distribution on Real Data')
ax1.set_xlabel('Confidence')
ax1.set_ylabel('Count')
ax1.legend()

# class breakdown
class_counts = {c: 0 for c in CLASS_NAMES}
for r in results:
    class_counts[r['class']] += 1

ax2.bar(range(N_CLASSES), list(class_counts.values()), color='steelblue', edgecolor='white', linewidth=0.5)
ax2.set_xticks(range(N_CLASSES))
ax2.set_xticklabels([c.replace('_', '
') for c in CLASS_NAMES], fontsize=7)
ax2.set_title('Predicted Class Counts (all windows)')
ax2.set_ylabel('Count')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures' / 'real_confidence_dist.png', dpi=120, bbox_inches='tight')
plt.show()

above = sum(1 for c in confs if c >= THRESHOLD)
print(f'Windows above threshold ({THRESHOLD}): {above}/{len(results)}  ({100*above/len(results):.1f}%)')
print(f'Mean confidence : {np.mean(confs):.3f}')
print(f'Median confidence: {np.median(confs):.3f}')


## Detections on Price Chart

We plot the S&P 500 price with detected patterns overlaid.
Only windows where confidence >= 0.55 are shown.
Green markers = bullish patterns, red = bearish.

In [ ]:
detections = [r for r in results if r['conf'] >= THRESHOLD and r['class'] != 'no_pattern']
print(f'Detections above threshold: {len(detections)}')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df['Close'], linewidth=0.8, color='steelblue', zorder=1)

for det in detections:
    color = 'green' if det['direction'] == 'bullish' else 'red'
    price = df.loc[det['date'], 'Close'] if det['date'] in df.index else None
    if price:
        ax.scatter(det['date'], price, color=color, s=60, zorder=2, alpha=0.8)

# legend
bull = mpatches.Patch(color='green', label='Bullish pattern')
bear = mpatches.Patch(color='red',   label='Bearish pattern')
ax.legend(handles=[bull, bear])
ax.set_title(f'S&P 500 with Pattern Detections (conf >= {THRESHOLD})', fontsize=12)
ax.set_ylabel('Price')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures' / 'real_detections.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# confidence over time — shows where the model is uncertain
fig, ax = plt.subplots(figsize=(14, 4))
det_dates = [r['date'] for r in results]
ax.plot(det_dates, confs, linewidth=0.6, color='steelblue', alpha=0.8)
ax.axhline(THRESHOLD, color='red', linestyle='--', linewidth=0.8, label=f'threshold={THRESHOLD}')
ax.axhline(1/N_CLASSES, color='gray', linestyle='--', linewidth=0.8, label='random (14.3%)')
ax.fill_between(det_dates, confs, THRESHOLD,
                where=[c >= THRESHOLD for c in confs],
                alpha=0.2, color='green', label='above threshold')
ax.set_title('Model Confidence Over Time (Real Data)', fontsize=12)
ax.set_ylabel('Confidence')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures' / 'real_confidence_time.png', dpi=120, bbox_inches='tight')
plt.show()


## Save Results for Notebook 05

We save the detections to a file so notebook 05 can load them for backtesting
without re-running the full sliding window inference.

In [ ]:
import json as json_lib

# save results as json for notebook 05
save_data = []
for r in results:
    save_data.append({
        'date':      str(r['date'].date()),
        'class':     r['class'],
        'direction': r['direction'],
        'conf':      float(r['conf']),
    })

out_path = RESULTS_DIR / 'detections.json'
with open(out_path, 'w') as f:
    json_lib.dump(save_data, f)

print(f'Saved {len(save_data)} detections to {out_path}')
print(f'Bullish detections : {sum(1 for r in save_data if r["direction"] == "bullish")}')
print(f'Bearish detections : {sum(1 for r in save_data if r["direction"] == "bearish")}')
print(f'No pattern         : {sum(1 for r in save_data if r["direction"] == "neutral")}')
